In [1]:
from gptnl_pii_mappers._backend.ner_pii_list import _print_classes_for_pii_ner
from gptnl_pii_mappers._backend.regex_pii_list import (
    _print_chain_formatters,
    _print_classes_for_pii_regex,
)

_print_classes_for_pii_regex()

print("\n\n")

_print_classes_for_pii_ner()

print("\n\n")

_print_chain_formatters()

PII_EmailAddress = regex_factory(**COMPUTER_REGEX['e-mail_address'])
PII_IPAddress = regex_factory(**COMPUTER_REGEX['ip_address'])
PII_URL = regex_factory(**COMPUTER_REGEX['url'])
PII_MACAddress = regex_factory(**COMPUTER_REGEX['mac_address'])
PII_FilePath = regex_factory(**COMPUTER_REGEX['file_path'])

PII_CreditCardNumber = regex_factory(**BANKING_REGEX['credit_card_number'])
PII_CreditCardCVVCode = regex_factory(**BANKING_REGEX['credit_card_cvv_code'])
PII_CreditCardExpiryDate = regex_factory(**BANKING_REGEX['credit_card_expiry_date'])
PII_IBAN = regex_factory(**BANKING_REGEX['iban'])
PII_EuropeanVATNumber = regex_factory(**BANKING_REGEX['european_vat_number'])

PII_InternationalPhoneNumber = regex_factory(**COMMON_REGEX['international_phone_number'])

PII_DutchLicensePlateNumber = regex_factory(**DUTCH_REGEX['dutch_license_plate_number'])
PII_DutchPhoneNumber = regex_factory(**DUTCH_REGEX['dutch_phone_number'])
PII_DutchIdentityNumberBSN = regex_factory(**DUTCH_REGEX['dutch_identit

In [2]:
import os

from gptnl_pii_mappers import (
    AUSTRALIAN_REGEX,
    BANKING_REGEX,
    BELGIAN_REGEX,
    CANADIAN_REGEX,
    COMMON_REGEX,
    COMPUTER_REGEX,
    DUTCH_REGEX,
    NER_FLAIR,
    UK_REGEX,
    US_REGEX,
    NER_GLiNER,
    NER_RobBERT,
    NER_XLM_RoBERTa,
)


def test_file_str(language, pii_class_name, pii_detection_type, detection_type_model):
    test_function_name = f"test_{language.lower()}_{pii_class_name.lower().replace('pii_', '')}"
    return f"""import pandas as pd
import pytest

from gptnl_pii_mappers import {pii_class_name}{detection_type_model}
from tests.conftest import SELECT_N_SAMPLES, evaluate_formatting

# Load the data and prepare tuples for parameterization
df = pd.read_csv("data/gptnl_synthetic_pii_data/{language}/{pii_class_name}.csv", sep=";", dtype=str)
testdata = [(row.Sentence, int(row.Label), row.Type, row.Entity) for row in df[["Sentence", "Label", "Type", "Entity"]].itertuples(index=False)]
testdata = testdata[:SELECT_N_SAMPLES]


@pytest.fixture(scope="class")
def formatter():
    \"\"\"Provides a {pii_class_name}{detection_type_model} formatter instance for testing.

    Yields:
        {pii_class_name}{detection_type_model}: An instance of the {pii_class_name}{detection_type_model} class for use in tests.
    \"\"\"
    formatter_instance = {pii_class_name}{detection_type_model}()
    yield formatter_instance
    del formatter_instance


@pytest.mark.parametrize("sentence, expected_label, test_type, entity", testdata)
class Test{language}{pii_class_name}:
    def {test_function_name}(self, formatter, sentence, expected_label, test_type, entity, request):
        formatted_sentence = formatter.format(sentence, language="{language.lower()}")

        predicted_label, evaluation_result = evaluate_formatting(expected_label, sentence, formatted_sentence, entity, formatter.mask)

        request.node.user_properties.append(("name", "{pii_class_name.replace('PII_', '')}"))
        request.node.user_properties.append(("model_name", "{detection_type_model.replace('_', '')}"))
        request.node.user_properties.append(("language", "{language}"))
        request.node.user_properties.append(("detection_type", "{pii_detection_type.upper()}"))
        request.node.user_properties.append(("test_type", test_type))
        request.node.user_properties.append(("expected", expected_label))
        request.node.user_properties.append(("predicted", predicted_label))
        request.node.user_properties.append(("sentence", sentence))
        request.node.user_properties.append(("formatted_sentence", formatted_sentence))
        request.node.user_properties.append(("entity", entity))
        request.node.user_properties.append(("mask", formatter.mask))
        request.node.user_properties.append(("metadata", ""))
        request.node.user_properties.append(("reason", evaluation_result))

        assert evaluation_result == "pass", evaluation_result
"""


def create_test_files(language, dictionary, pii_detection_type, detection_type_model=""):
    os.makedirs(folder_path := f"tests/{language.lower()}/{pii_detection_type}", exist_ok=True)
    with open(f"{folder_path}/__init__.py", "w"):
        pass

    with open(f"tests/{language.lower()}/__init__.py", "w"):
        pass

    for ner_info in dictionary.values():
        name = ner_info["name"]
        if "PersonAndOrganisation" in name:
            continue

        with open(f"{folder_path}/{name.lower()}{detection_type_model}_test.py", "w") as f:
            f.write(test_file_str(language, name, pii_detection_type, detection_type_model))


create_test_files("NL", NER_GLiNER, "ner", "_GLiNER")
create_test_files("EN", NER_GLiNER, "ner", "_GLiNER")

create_test_files("NL", NER_FLAIR, "ner", "_Flair_Large")
create_test_files("EN", NER_FLAIR, "ner", "_Flair_Large")

create_test_files("NL", NER_RobBERT, "ner", "_RobBERT")

create_test_files("NL", NER_XLM_RoBERTa, "ner", "_XLM_RoBERTa")

NL_REGEX = {}
NL_REGEX.update(BELGIAN_REGEX)
NL_REGEX.update(DUTCH_REGEX)
NL_REGEX.update(BANKING_REGEX)
NL_REGEX.update(COMPUTER_REGEX)
NL_REGEX.update(COMMON_REGEX)

EN_REGEX = {}
EN_REGEX.update(AUSTRALIAN_REGEX)
EN_REGEX.update(CANADIAN_REGEX)
EN_REGEX.update(UK_REGEX)
EN_REGEX.update(US_REGEX)
EN_REGEX.update(BANKING_REGEX)
EN_REGEX.update(COMPUTER_REGEX)
EN_REGEX.update(COMMON_REGEX)

create_test_files("NL", NL_REGEX, "regex")

create_test_files("EN", EN_REGEX, "regex")

In [3]:
everything_on_all_test_file = """from pathlib import Path

import pandas as pd
import pytest

from gptnl_pii_mappers import NAME_TO_MASK, PII_Everything

from .conftest import SELECT_N_SAMPLES, evaluate_formatting

dutch_csv_files = list(Path("data/gptnl_synthetic_pii_data/NL").glob("*.csv"))
english_csv_files = list(Path("data/gptnl_synthetic_pii_data/EN").glob("*.csv"))

testdata = []

for csv_file in dutch_csv_files + english_csv_files:
    # Load the CSV file
    df = pd.read_csv(csv_file, sep=";", dtype=str)
    name = csv_file.name.replace(".csv", "")
    if "PersonAndOrganisation" in name:
        continue

    mask = NAME_TO_MASK[name]
    language = str(csv_file).split("/")[-2]
    # Prepare test data tuples from the dataframe
    pii_specific_testdata = [(name.replace("PII_", ""), row.Sentence, int(row.Label), row.Type, row.Entity, language, mask) for row in df[["Sentence", "Label", "Type", "Entity"]].itertuples(index=False)]
    pii_specific_testdata = pii_specific_testdata[:SELECT_N_SAMPLES]
    testdata.extend(pii_specific_testdata)


# Fixture for setup and teardown
@pytest.fixture(scope="class")
def formatter():
    \"\"\"Provides a PII_Everything formatter instance for testing.

    Yields:
        PII_Everything: An instance of the PII_Everything class for use in tests.
    \"\"\"
    formatter_instance = PII_Everything()
    yield formatter_instance
    del formatter_instance


@pytest.mark.parametrize("name, sentence, expected_label, test_type, entity, language, mask", testdata)
class TestEverythingOnAll:
    def test_everything_on_all(self, formatter, name, sentence, expected_label, test_type, entity, language, mask, request):  # noqa: PLR0913
        formatted_sentence, metadata = formatter.format(sentence, language=language.lower(), with_metadata=True)

        predicted_label, evaluation_result = evaluate_formatting(expected_label, sentence, formatted_sentence, entity, mask)

        request.node.user_properties.append(("name", name))
        request.node.user_properties.append(("model_name", "'Best'"))
        request.node.user_properties.append(("language", language))
        request.node.user_properties.append(("detection_type", "Everything"))
        request.node.user_properties.append(("test_type", test_type))
        request.node.user_properties.append(("expected", expected_label))
        request.node.user_properties.append(("predicted", predicted_label))
        request.node.user_properties.append(("sentence", sentence))
        request.node.user_properties.append(("formatted_sentence", formatted_sentence))
        request.node.user_properties.append(("entity", entity))
        request.node.user_properties.append(("mask", mask))
        request.node.user_properties.append(("metadata", metadata))
        request.node.user_properties.append(("reason", evaluation_result))

        assert evaluation_result == "pass", evaluation_result
"""

def create_everything_on_all_test(folder_path = 'tests', file_name = 'pii_everything_on_all_test.py'):
    os.makedirs(folder_path, exist_ok=True)

    # Write the content to the file
    with open(os.path.join(folder_path, file_name), 'w') as file:
        file.write(everything_on_all_test_file)

    print(f"Code has been written to {os.path.join(folder_path, file_name)}")

# Usage
create_everything_on_all_test()

Code has been written to tests/pii_everything_on_all_test.py
